In [3]:
!pip install -q sentence-transformers



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
from sentence_transformers import SentenceTransformer
import numpy as np

#  Step 1: Load JSONL data with UTF-8 decoding
data = []
with open('C:\\ReWorld\\Sanjay\\supercleaned_for_embedding.jsonl', 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("⚠️ Skipping bad line:", e)

#  Step 2: Prepare input strings for embedding
fields_to_embed = ['Funder', 'grant_type', 'funding_focus', 'Funding Amount', 'Application Deadline', 'org_type','geographic_focus']
sentences = []

for item in data:
    parts = [str(item.get(field, '')).strip() for field in fields_to_embed]
    combined_text = " | ".join(parts)
    sentences.append(combined_text)

#  Step 3: Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

#  Step 4: Generate embeddings
embeddings = model.encode(sentences, show_progress_bar=True)
print(f"✅ Embeddings created. Shape: {embeddings.shape}")

#  Step 5: Add embeddings back to each item
for i, item in enumerate(data):
    item['embedding'] = embeddings[i].tolist()

#  Step 6: Save as JSONL with embeddings
output_path = 'data_with_embeddings.jsonl'
with open(output_path, 'w', encoding='utf-8') as file:
    for item in data:
        json.dump(item, file)
        file.write('\n')

print(f"✅ Data with embeddings saved as '{output_path}'")

#  Step 7: Save embeddings to a new JSONL file
output_path = 'embeddings_only.jsonl'
with open(output_path, 'w', encoding='utf-8') as file:
    for emb in embeddings:
        json.dump({'embedding': emb.tolist()}, file)
        file.write('\n')

print(f"✅ Embeddings-only file saved as '{output_path}'")



Batches: 100%|██████████| 3/3 [00:01<00:00,  1.92it/s]

✅ Embeddings created. Shape: (88, 384)
✅ Data with embeddings saved as 'data_with_embeddings.jsonl'
✅ Embeddings-only file saved as 'embeddings_only.jsonl'
